## Phần 0 — Khởi động (5 phút)

- [ ] **0.1 Seed và device.** Viết hàm `seed_everything(seed)` đặt seed cho `random`, `numpy`, `torch` (CPU + CUDA). Tạo biến `device` tự chọn `cuda` nếu có, không thì `cpu`.
  - **Kiểm tra:** gọi `seed_everything(42)` hai lần, mỗi lần in `torch.rand(3)` → hai kết quả giống hệt nhau.

<details><summary>Gợi ý cú pháp</summary>

`random.seed`, `np.random.seed`, `torch.manual_seed`, `torch.cuda.manual_seed_all`, `torch.cuda.is_available()`

In [1]:
import numpy as np
import torch
import random
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import ToTensor
import torchvision.transforms as T
import albumentations as A 
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import torchvision
import torch.nn as nn
import torch.optim as optim

/Users/quangmanh/Project/Final-competitive/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/quangmanh/Project/Final-competitive/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def seed_everything(seed: int):
    ''' seed python before
    seed random lib in python
    seed numpy 
    seed torch
    seed cuda 
    seed compute in GPU
    ** all things in python - becuz it is independence
    =====================
    params: seed (int)
    return device - seed '''

    os.environ['PYTHONASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        device = torch.device('cuda')
        torch.cuda.seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(torch.rand((5, 4)))
    print(device)
    return device
    

In [3]:
seed_everything(42)

tensor([[0.8823, 0.9150, 0.3829, 0.9593],
        [0.3904, 0.6009, 0.2566, 0.7936],
        [0.9408, 0.1332, 0.9346, 0.5936],
        [0.8694, 0.5677, 0.7411, 0.4294],
        [0.8854, 0.5739, 0.2666, 0.6274]])
mps


device(type='mps')

**1.1 Đọc nhãn.** Đọc `train.csv` bằng pandas. In số ảnh, số lớp, số ảnh mỗi lớp. Tạo `class2idx` và `idx2class`.

- **Kiểm tra:** `len(class2idx) == 6`; `idx2class[class2idx['glass']] == 'glass'`.

In [4]:
train_dir = '/Users/quangmanh/Project/Final-competitive/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/annotations.csv'
df = pd.read_csv(train_dir)
print(df.shape)
df.head()

(8385, 3)


,file_name,is_negative,category_id
0,201508010000_1513_8f8e20b8.jpg,False,1.0
1,201508010000_1513_9d8a4002.jpg,True,NaN
2,201508010300_1513_14239637.jpg,False,1.0
3,201508010300_1513_c7793163.jpg,True,NaN
4,201508010600_1513_a80d2e75.jpg,False,2.0


In [5]:
df_neg = df[df['is_negative'] == False]
df.shape[0] - df_neg.shape[0]

2548

In [6]:
df['category_id'].value_counts(dropna=False)

category_id
NaN    2548
1.0    1931
2.0    1583
4.0    1553
3.0     770
Name: count, dtype: int64

In [7]:
CLASSES = ['no_storm', 'cat1', 'cat2', 'cat3', 'cat4']

In [8]:
{ idx: cls_name for idx, cls_name in enumerate(CLASSES)}
{ cls_name: idx for idx, cls_name in enumerate(CLASSES)}

{'no_storm': 0, 'cat1': 1, 'cat2': 2, 'cat3': 3, 'cat4': 4}

In [9]:
df['label'] = df['category_id'].fillna(0).astype(int)
df.head()

,file_name,is_negative,category_id,label
0,201508010000_1513_8f8e20b8.jpg,False,1.0,1
1,201508010000_1513_9d8a4002.jpg,True,NaN,0
2,201508010300_1513_14239637.jpg,False,1.0,1
3,201508010300_1513_c7793163.jpg,True,NaN,0
4,201508010600_1513_a80d2e75.jpg,False,2.0,2


In [10]:
df['storm_id'] = df['file_name'].str.split('_').str[1]
df['year'] = df['storm_id'].str.slice(0, 2)
df.head()

,file_name,is_negative,category_id,label,storm_id,year
0,201508010000_1513_8f8e20b8.jpg,False,1.0,1,1513,15
1,201508010000_1513_9d8a4002.jpg,True,NaN,0,1513,15
2,201508010300_1513_14239637.jpg,False,1.0,1,1513,15
3,201508010300_1513_c7793163.jpg,True,NaN,0,1513,15
4,201508010600_1513_a80d2e75.jpg,False,2.0,2,1513,15


In [11]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, val_idx = next(sgkf.split(df, df.label, groups=df.storm_id))

In [12]:
train_split, val_split = train_test_split(df, random_state=42, stratify=df.label)

In [13]:
class ImageDataset(Dataset):
    def __init__(self, df, img_dir, transform = False, has_label=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.has_label = has_label
        self.file_name = self.df.iloc[:,0]
        if self.has_label:
            self.labels = self.df.iloc[:,3]
    def __len__(self):
        return len(self.df)
    def __getitem__(self, index):
        img_name = self.file_name[index]
        img_path = os.join(self.img_dir, img_name)

        image = Image.open(img_path).convert('RGB').resize((224,224))
        image.save(os.join('/Users/quangmanh/Documents/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/images_resize', img_name))

        if self.transform:
            image = self.transform(image)
        else:
            image = ToTensor()(image)
        if self.has_label:
            label = self.labels[index]
            return image, torch.tensor(label, dtype=torch.long)
        return image, img_name

In [14]:
def make_test_df(img_dir):
    valid_format = ('.png', '.jpg', '.jpeg')

    list_image = sorted(os.listdir(img_dir))

    img_name = [
        f for f in list_image if f.lower().endswith(valid_format) and os.path.isfile(os.path.join(img_dir, f))
    ]

    test_df = pd.DataFrame({'file_name': img_name})
    return test_df

In [15]:
img_dir = '/Users/quangmanh/Documents/Olympic-AI-Practice-1/đề code tay/data/data-2/dataset/train/images'
train_dataset = ImageDataset(df, img_dir=img_dir, has_label=True)

In [16]:
dl_tr = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True, pin_memory=True)

In [17]:
num_sample = df.shape[0]
num_cls = len(df.label.unique())
df_num = df.label.value_counts().sort_index().reset_index()
count_c = (df_num['count']).to_numpy()
print(count_c)
# --------
class_weight = num_sample/(num_cls * count_c)
class_weight.tolist()

[2548 1931 1583  770 1553]


[0.6581632653061225,
 0.8684619368203004,
 1.0593809222994315,
 2.177922077922078,
 1.0798454603992274]

In [18]:
sample_weight = [class_weight[label] for label in df['label']]
sampler = torch.utils.data.WeightedRandomSampler(weights=sample_weight, num_samples=len(sample_weight), replacement=True)

In [19]:
df['fold'] = -1
stkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(stkf.split(df, df.label, groups=df.storm_id)):
    df.loc[val_idx, 'fold'] = fold_idx
print(df['fold'].value_counts())

fold
4    2058
2    1664
1    1561
3    1553
0    1549
Name: count, dtype: int64


In [20]:
train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop((224, 224), scale=(0.8, 1)),
    T.RandomRotation(180),
    T.ColorJitter(brightness=0.15, contrast=0.1),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [21]:
val_transform = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [22]:
train_transform_abl = A.Compose([
    A.Resize(256, 256),
    A.RandomResizedCrop((224, 224), scale=(0.8, 1)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.15, contrast=0.1),
    A.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ToTensorV2()
])

In [23]:
torch.save(torchvision.models.resnet18(weights='DEFAULT').state_dict(), 'resnet_18.pth')

In [24]:
device = torch.device('mps')
load_model = torch.load('resnet_18.pth', map_location=device)
if isinstance(load_model, dict): 
    if 'model_state_dict' in load_model:
        state_dict = load_model['model_state_dict']
        print('object model combination')
    else:
        print('Pure')
        state_dict = load_model
    model = torchvision.models.resnet18(weights=None)
    model.load_state_dict(state_dict)
    print(model)
else:
    model = load_model

model.to(device='mps')

Pure
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=Tru

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [25]:
def build_model(name, num_classes, weight_path):
    model = getattr(torchvision.models, name)(weights=None)
    in_feat = model.fc.in_features
    model.fc = nn.Linear(in_feat, num_classes)
    return model
model = build_model('resnet18', 5, weight_path='resnet_18.pth')
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [26]:
NUM_EPOCH = 5
print(torch.backends.mps.is_available())
device = torch.device('mps')

True


In [27]:
criterion = nn.CrossEntropyLoss()

In [28]:
print(f'Total params of model: {sum(p.numel() for p in model.parameters())}')

print(f'Training loop is starting with: {NUM_EPOCH} epoch')
for epoch in range(NUM_EPOCH):
    if epoch == 0:
        for name, param in model.named_parameters():
            if 'fc' in name:
                param.requires_grad = True 
            else:
                param.requires_grad =  False
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
        print(f'Total params of model: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')
    elif epoch == 2:
        for param in model.parameters():
            param.requires_grad = True 
        optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    model.train()


Total params of model: 11179077
Training loop is starting with: 5 epoch
Total params of model: 2565


In [29]:
back_bone_params = []
head_param = []

for name, param in model.named_parameters():
    if param.requires_grad:
        if 'fc' in name:
            head_param.append(param)
        else: 
            back_bone_params.append(param)
optimizer = optim.AdamW([
    {'params': back_bone_params, 'lr': 1e-4},
    {'params': head_param, 'lr': 1e-3}
])

In [30]:
class SimpleCNN:
    def __init__(self, num_class):
        super(SimpleCNN, self).__init__()

        self.features_extract = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride = 1, padding=1),
            nn.BatchNorm2d(),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.global_pool = nn.AvgPool2d((1,1))
        self.flatten = nn.Flatten()
        self.header = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, num_class)
        )

    def forward(self, x):  
        x = self.features_extract(x)
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.header(x)
        return x

In [31]:
class BasicBlock:
    expansion = 1
    def __init__(self, input_channel, output_channel, stride = 1, downsample = None):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(input_channel, output_channel, stride, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(output_channel)

        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(output_channel, output_channel, stride, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(output_channel)

        self.downsample = downsample

    
    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity

        out = self.relu(out)
        return out
class Resnet18():
    def __init__(self, num_class):
        super(Resnet18, self).__init__()
        self.in_channel = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.relu = nn.ReLU()

        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self._layer1 = self._make_layers(64, 2, stride=1)
        self._layer2 = self._make_layers(128, 2, stride=1)
        self._layer3 = self._make_layers(256, 2, stride=1)
        self._layer4 = self._make_layers(512, 2, stride=1)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_class)
    def _make_layers(self, output_channel, blocks, stride = 1):
        downsample = None

        if stride != 1 or self.in_channel != output_channel:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channel, output_channel, stride=stride, kernel_size=1, bias=False)
            )
        layers = []
        layers.append(BasicBlock(self.in_channel, output_channel, stride, downsample))
        self.in_channel = output_channel
        for _, i in range(1, blocks):
            layers.append(BasicBlock(self.in_channel, output_channel, ))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self._layer1(x)
        x = self._layer2(x)
        x = self._layer3(x)
        x = self._layer4(x)

        x = self.global_pool(x)
        x = torch.flatten(x)
        x = self.fc(x)

        return x

In [38]:
class DenseLayer():
    def __init__(self, in_channel, growrate):
      super(DenseLayer, self).__init__()

      self.bn1 = nn.BatchNorm2d(in_channel)
      self.relu = nn.ReLU()
      self.conv1 = nn.Conv2d(in_channel, 4*growrate, kernel_size=1, stride=1, bias=False)
      self.bn2 = nn.BatchNorm2d(4*growrate)
      self.conv2 = nn.Conv2d(4*growrate, growrate, kernel_size=3, stride=1, bias=False)

    def forward(self, init_feat):
        if isinstance(init_feat, torch.Tensor):
          init_feat = [init_feat]
        x = torch.concat(init_feat, dim = 1)

        out = self.bn1(x)
        out = self.relu(out)
        out = self.conv1(out)

        out = self.bn2(out)
        out = self.relu(out)
        new_feat = self.conv2(out)
        return new_feat

class DenseBlock:
   def __init__(self, num_layer, in_channel, growrate):
      super(DenseBlock, self).__init__()

      for i in range(num_layer):
         layer = DenseLayer(in_channel + i*growrate, growrate)
         self.add_module(f'dense_layer {i+1}', layer)
   def forward(self, init_feat):
      feats = [init_feat]
      for name, layer in self.items():
         new_feat = layer(feats)
         feats.append(new_feat)
      return torch.cat(feats, dim = 1)
class Transition:
   def __init__(self, in_channel, out_channel):
      super(Transition, self).__init__()
      self.bn = nn.BatchNorm2d(in_channel)
      self.relu = nn.ReLU()
      self.conv1 = nn.Conv2d(in_channel, out_channel, kernel_size=1, stride=1, bias=False)
      self.avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
   def forward(self, x):
      out = self.bn(x)
      out = self.relu(out)
      out = self.conv1(out)
      out = self.avg_pool(out)

      return out

class Densenet121:
  def __init__(self, num_class, growrate, reduction):
      super(Densenet121, self).__init__()
      num_init_feat = 64

      self.features = nn.Sequential(
         nn.Conv2d(3, num_init_feat, kernel_size=7, stride=2, padding=3, bias=False),
         nn.BatchNorm2d(num_init_feat),
         nn.ReLU(),
         nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
      )
      num_layer_per_block = [6, 12, 24, 16]
      in_channels = num_init_feat

      for i, num_layer in enumerate(num_layer_per_block):
         block = DenseBlock(num_layer, in_channels, growrate)
         self.features.add_module(f'DenseBlock {i+1}', block)

         in_channels = in_channels + growrate * num_layer

         if i != (len(num_layer_per_block) -1):
            out_channel = int(in_channels * reduction)
            transition = Transition(in_channels, out_channel)
            self.features.add_module(f'transition {i+1}', transition)
            in_channels = out_channel
      self.bn = nn.BatchNorm2d(in_channels)
      self.relu = nn.ReLU(inplace=True)
      self.avg_pool = nn.AvgPool2d((1, 1))
      self.classifier = nn.Linear(in_channels, num_class)

  def forward(self, x):
     x = self.features(x)
     x = self.bn(x)
     x = self.relu(x)
     x = self.avg_pool(x)
     x = nn.Flatten(x, 1)
     x = self.classifier(x)
     return x

In [43]:
class SqueezeExcite:
    def __init__(self, ch, se_ch):
        super(SqueezeExcite, self).__init__()

        self.se = nn.Sequential(
            nn.AvgPool2d((1, 1)),
            nn.Conv2d(ch, se_ch, kernel_size=1),
            nn.SiLU(),
            nn.Conv2d(se_ch, ch, kernel_size=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se[1]

class MBConv:
    def __init__(self, in_channel, out_channel, kernel, stride, expand):
        super(MBConv, self).__init__()

        self.expand = expand
        self.stride = stride
        self.in_channel = in_channel
        self.out_channel = out_channel

        self.hidden_channel = expand*in_channel

        self.conv1 = nn.Conv2d(in_channel, self.hidden_channel, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(self.hidden_channel)
        self.silu = nn.SiLU(inplace=True)
        self.depth_conv = nn.Conv2d(
            self.hidden_channel, self.hidden_channel,
            kernel_size=kernel, stride=stride, padding=kernel//2,
            groups=self.hidden_channel, bias=False
        )
        self.bn2 = nn.BatchNorm2d(self.hidden_channel)
        se_ch = max(1, in_channel//4)
        self.se = SqueezeExcite(self.hidden_channel, se_ch=se_ch)

        self.conv2 = nn.Conv2d(self.hidden_channel, out_channel, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channel)

    def forward(self, x):
        identity = x

        if self.expand != 1:
            x = self.conv1(x)
        x = self.bn1(x)
        x = self.silu(x)
        x = self.depth_conv(x)
        x = self.bn2(x)
        x = self.silu(x)
        
        x = self.se(x)
        
        x = self.conv2(x)
        x = self.bn3(x)
        if self.stride == 1 and self.in_channel == self.out_channel:
            x += identity
        return x
    
class EffcientNetB0:
    def __init__(self, num_class):
        super(EffcientNetB0, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True)
        )

        cur_in_channel = 32

        b0_config = [
            (1, 3, 1, 16,  1),
            (6, 3, 2, 24,  2),
            (6, 5, 2, 40,  2),
            (6, 3, 2, 80,  3),
            (6, 5, 1, 112, 3),
            (6, 5, 2, 192, 4),
            (6, 3, 1, 320, 1),
        ]

        layers = []
        for expand, kernel, stride, out, repeat in b0_config:
            for i in range(repeat):
                layer_stride = stride if i == 0 else 1
                layers.append(MBConv(
                    cur_in_channel,
                    out,
                    kernel,
                    layer_stride,
                    expand
                ))
                cur_in_channel = out 
        self.blocks = nn.Sequential(*layers)

        self.head_conv = nn.Sequential(
            nn.Conv2d(cur_in_channel, 1280, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU(inplace=True)
        )
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.2)
        self.classifier = nn.Linear(1280, num_class)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head_conv(x)
        x = self.avgpool(x)
        x = nn.Flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)

        return x